# Phase 7–12 — Model Training, Evaluation, News Sentiment, Backtest and Risk Level

Runs the whole modelling half of the assignment:

* **Phase 7** — train Logistic Regression, Random Forest, Gradient Boosting and XGBoost
  (time-series cross-validation, no shuffle).
* **Phase 8** — accuracy / precision / recall / F1 / ROC-AUC / confusion matrix + comparison
  table.
* **Phase 9** — VADER sentiment over the collected headlines.
* **Phase 10** — retrain **with** the sentiment features and compare.
* **Phase 11** — backtest the selected model over the held-out window.
* **Phase 12** — volatility (risk) indicator.

> Expect ~1 minute for Phase 7 and ~1 minute for Phase 10.

In [1]:
from pathlib import Path
import sys

# make the project root importable no matter where the notebook is started from
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 60)

from src.features import BASE_FEATURES, TARGET, build_dataset, make_split
from src.modeling import train_models, select_best, REPORTS_DIR
import src.pipeline as pipeline

full = build_dataset()
train, test = make_split(full)
print(f"train {len(train)} rows | test {len(test)} rows")

Loaded cached data <- C:\Users\KIIT\Documents\StockMarketAnalyzer\stock_market_ai\data\stock_data.csv

--- Phase 1 cleaning report ---
  missing_before: {}
  duplicate_rows: 0
  missing_after: {}
  rows: 6480
  symbols: ['RELIANCE.NS', 'TCS.NS', '^NSEI']
  date_min: 2018-01-01
  date_max: 2026-09-28
Feature matrix: 6480 rows -> 6420 usable rows (warm-up/target rows dropped: 60)
train 5138 rows | test 1282 rows


## Phase 7 — Model development (three models required, four trained)

In [2]:
comparison, models = train_models(train, test, list(BASE_FEATURES))
comparison


=== Logistic Regression ===


  params={'C': 0.01}
  accuracy=0.4875  f1=0.5807  auc=0.5094
  confusion matrix:
[[170 508]
 [149 455]]

=== Random Forest ===


  params={'max_depth': 6, 'min_samples_leaf': 1}
  accuracy=0.4930  f1=0.5932  auc=0.5137
  confusion matrix:
[[158 520]
 [130 474]]

=== Gradient Boosting ===


  params={'learning_rate': 0.05, 'max_depth': 5}
  accuracy=0.5078  f1=0.5150  auc=0.5083
  confusion matrix:
[[316 362]
 [269 335]]

=== XGBoost ===


  params={'max_depth': 3}
  accuracy=0.5039  f1=0.5310  auc=0.5067
  confusion matrix:
[[286 392]
 [244 360]]


,Model,Accuracy,Precision,Recall,F1,ROC-AUC,BestParams
0,Logistic Regression,0.49,0.47,0.75,0.58,0.51,{'C': 0.01}
1,Random Forest,0.49,0.48,0.78,0.59,0.51,"{'max_depth': 6, 'min_samples_leaf': 1}"
2,Gradient Boosting,0.51,0.48,0.55,0.52,0.51,"{'learning_rate': 0.05, 'max_depth': 5}"
3,XGBoost,0.50,0.48,0.60,0.53,0.51,{'max_depth': 3}
4,Baseline (majority class),0.47,0.47,1.00,0.64,NaN,{}


In [3]:
# confusion matrices for every trained model
from src.pipeline import plot_confusion
for name, tm in models.items():
    display(plot_confusion(tm, f"confusion_matrix_{name.replace(' ', '_').lower()}.png"))

WindowsPath('C:/Users/KIIT/Documents/StockMarketAnalyzer/stock_market_ai/reports/confusion_matrix_logistic_regression.png')

WindowsPath('C:/Users/KIIT/Documents/StockMarketAnalyzer/stock_market_ai/reports/confusion_matrix_random_forest.png')

WindowsPath('C:/Users/KIIT/Documents/StockMarketAnalyzer/stock_market_ai/reports/confusion_matrix_gradient_boosting.png')

WindowsPath('C:/Users/KIIT/Documents/StockMarketAnalyzer/stock_market_ai/reports/confusion_matrix_xgboost.png')

In [4]:
# feature importance of the selected model
from src.pipeline import plot_importance
best = select_best(comparison)
imp = models[best].importance.sort_values(ascending=False)
display(imp.round(4).to_frame("importance"))
display(plot_importance(models[best]))

,importance
Volume_Change,0.06
Daily_Return,0.06
Close_vs_MA20,0.06
Previous_Day_Return,0.05
Return_Mean_5,0.05
Close_Open_Chg,0.05
Return_Lag_2,0.05
Return_Lag_3,0.05
Close_vs_MA5,0.04
Volatility_10,0.04


WindowsPath('C:/Users/KIIT/Documents/StockMarketAnalyzer/stock_market_ai/reports/feature_importance.png')

### Phase 8 answers

**A. Which model performed better?** Look at the table above — the winner is chosen on the
**F1** score on the untouched last-20 % window (see `reports/model_comparison.csv` for the
full, reproducible numbers).

**B. Why was that model selected?** F1 balances precision and recall, which matters here
because the classes are close to 50/50 and a model can otherwise win by predicting "UP"
every day. Accuracy alone would reward that degenerate strategy. The comparison table also
reports the *majority-class baseline* so the gain over doing nothing is explicit.

**C. Does high accuracy mean the model will be profitable?** **No.** Accuracy measures
direction only, not magnitude: a model can be right on 51 % of days that move ±0.05 % and
wrong on the few 5 % days that determine the P&L, transaction costs and slippage are
ignored, and accuracy is computed on a regime that may not repeat. Phase 11 therefore also
plots a strategy equity curve against buy-and-hold, which is the honest test.

## Phase 9 — Financial news sentiment

Headlines are collected from the **GDELT Document API** (`python -m src.collect_news`) into
`data/news_data.csv`, then scored with **VADER**:

```
compound >=  0.05 -> Positive -> +1
compound <= -0.05 -> Negative -> -1
otherwise         -> Neutral  ->  0
```

**A. How can financial news affect stock prices?** News changes expectations about future
cash flows (earnings, guidance, contracts, regulation) and about risk (rates, geopolitics,
pandemics), so rational repricing moves the price; big headlines also trigger herd
behaviour and forced flows, which amplifies the move. Company-specific news moves that
stock, macro news moves the whole index.

**B. Limitations of general sentiment models on financial news?** VADER was trained on
informal social-media text, not financial reports: it ignores numbers ("profits fell 20 %"
scores positive because of "profits"), it misreads negation and domain words ("upper beat
guidance", "tax loss", "short squeeze"), it cannot separate a company from the sector, and
a headline's market impact depends on whether it was already priced in — which no
bag-of-words score can know.

In [5]:
from src.sentiment import load_news, sentiment_report, score_dataframe, daily_sentiment

news = load_news()
print(f"headlines: {len(news)}  |  {news['Date'].min().date()} -> {news['Date'].max().date()}")
display(sentiment_report(news))

sample = score_dataframe(news)
display(sample.groupby("label")["compound"].describe().round(3))
display(sample[["Date", "Symbol", "Headline", "label", "compound"]]
        .sort_values("compound").head(5))       # most negative
display(sample[["Date", "Symbol", "Headline", "label", "compound"]]
        .sort_values("compound", ascending=False).head(5))   # most positive

headlines: 2979  |  2018-10-03 -> 2026-07-20


label,Negative,Neutral,Positive
Symbol,,,
RELIANCE.NS,0.20,0.32,0.48
TCS.NS,0.24,0.31,0.45
^NSEI,0.15,0.22,0.63


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
Negative,615.00,-0.44,0.22,-0.94,-0.61,-0.42,-0.27,-0.05
Neutral,910.00,0.00,0.01,-0.03,0.00,0.00,0.00,0.03
Positive,"1,454.00",0.50,0.22,0.05,0.34,0.49,0.68,0.96


,Date,Symbol,Headline,label,compound
1877,2018-10-04,TCS.NS,TCS employee fired for sending rape and death ...,Negative,-0.94
1206,2024-10-04,RELIANCE.NS,Mkts suffer worst week in over 2 yrs ; West As...,Negative,-0.91
1909,2018-10-05,TCS.NS,"TCS sacks employee for sending rape , death th...",Negative,-0.91
1901,2018-10-04,TCS.NS,TCS Staffer Sacked For Sending Rape & Death Th...,Negative,-0.91
1889,2018-10-04,TCS.NS,TCS fires employee for sending rape and death ...,Negative,-0.91


,Date,Symbol,Headline,label,compound
1875,2018-10-04,TCS.NS,45 Organizations Win Prestigious ATD BEST Awar...,Positive,0.96
1050,2023-10-12,RELIANCE.NS,2023 Gulf Energy Information Excellence Award ...,Positive,0.95
1294,2024-10-10,RELIANCE.NS,The Gulf Energy Information Excellence Awards ...,Positive,0.94
1525,2025-10-17,RELIANCE.NS,The Gulf Energy Information Excellence Awards ...,Positive,0.94
1540,2025-10-17,RELIANCE.NS,Upstream winners honored at 2025 Gulf Energy I...,Positive,0.94


## Phase 10 — Combine news and market features

The daily news score is aggregated over a **5-day trailing window and lagged by one day**
(`src/sentiment.py`), so the feature for day *t* only contains headlines published on
*t−1* or earlier. News published after the prediction date can never be seen.

**A. Did sentiment improve performance?** Compare the two tables below
(`reports/sentiment_impact.csv`).

**B. If not, why?** Typical reasons: daily aggregation dilutes a single headline across
five sessions; most collected headlines are macro/index news rather than news about the
selected stock on that exact day; prices often move on *expectations* rather than on the
verbal tone of a headline; and a neutral score carries no directional information at all.
A general-purpose sentiment model also mismeasures financial language (Phase 9B).

In [6]:
from src.sentiment import add_sentiment_to_features
feat_news = add_sentiment_to_features(full, load_news(), window_days=5, lag=1)
train_n, test_n = make_split(feat_news)

feats = list(BASE_FEATURES) + ["Headlines", "News_Sentiment_Compound",
                               "News_Sentiment_Mean_Value", "News_Sentiment"]
comparison_news, models_news = train_models(train_n, test_n, feats)

impact = (comparison[["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]]
          .merge(comparison_news[["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]],
                 on="Model", suffixes=("_no_news", "_with_news")))
for m in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
    impact["delta_" + m] = (impact[m + "_with_news"] - impact[m + "_no_news"]).round(4)
display(impact)


=== Logistic Regression ===


  params={'C': 0.01}
  accuracy=0.4891  f1=0.5820  auc=0.5135
  confusion matrix:
[[171 507]
 [148 456]]

=== Random Forest ===


  params={'max_depth': 6, 'min_samples_leaf': 1}
  accuracy=0.4906  f1=0.5875  auc=0.5088
  confusion matrix:
[[164 514]
 [139 465]]

=== Gradient Boosting ===


  params={'learning_rate': 0.1, 'max_depth': 5}
  accuracy=0.4984  f1=0.4965  auc=0.5004
  confusion matrix:
[[322 356]
 [287 317]]

=== XGBoost ===


  params={'max_depth': 3}
  accuracy=0.4891  f1=0.5222  auc=0.5035
  confusion matrix:
[[269 409]
 [246 358]]


,Model,Accuracy_no_news,Precision_no_news,Recall_no_news,F1_no_news,ROC-AUC_no_news,Accuracy_with_news,Precision_with_news,Recall_with_news,F1_with_news,ROC-AUC_with_news,delta_Accuracy,delta_Precision,delta_Recall,delta_F1,delta_ROC-AUC
0,Logistic Regression,0.49,0.47,0.75,0.58,0.51,0.49,0.47,0.76,0.58,0.51,0.00,0.00,0.00,0.00,0.00
1,Random Forest,0.49,0.48,0.78,0.59,0.51,0.49,0.47,0.77,0.59,0.51,-0.00,-0.00,-0.01,-0.01,-0.00
2,Gradient Boosting,0.51,0.48,0.55,0.52,0.51,0.50,0.47,0.52,0.50,0.50,-0.01,-0.01,-0.03,-0.02,-0.01
3,XGBoost,0.50,0.48,0.60,0.53,0.51,0.49,0.47,0.59,0.52,0.50,-0.01,-0.01,-0.00,-0.01,-0.00
4,Baseline (majority class),0.47,0.47,1.00,0.64,NaN,0.47,0.47,1.00,0.64,NaN,0.00,0.00,0.00,0.00,NaN


## Phase 11 — Backtest

Predicted vs. actual direction over the historical test period, with the Phase-11 counts.

In [7]:
import json
from src.backtest import run_backtest, plot_actual_vs_predicted, plot_cumulative_returns

all_cmp = pd.concat([comparison.assign(Experiment="without news"),
                     comparison_news.assign(Experiment="with news")])
best_name = select_best(all_cmp)
best_model = (models_news if best_name in models_news else models)[best_name]
feats_final = feats if best_name in models_news else list(BASE_FEATURES)
eval_df = test_n if best_name in models_news else test

stats, preds = run_backtest(eval_df, best_model, feats_final, save_plots=True)
display(pd.Series(stats))
preds[["Date", "Symbol", "Close", "Actual", "Predicted", "Confidence"]].head(10)

Saved C:\Users\KIIT\Documents\StockMarketAnalyzer\stock_market_ai\reports\backtest_actual_vs_predicted.png
Saved C:\Users\KIIT\Documents\StockMarketAnalyzer\stock_market_ai\reports\backtest_cumulative_return.png


Total Predictions        1,282.00
Correct Predictions        629.00
Incorrect Predictions      653.00
UP Predictions             979.00
DOWN Predictions           303.00
Actual UP Days             604.00
Actual DOWN Days           678.00
Directional Accuracy         0.49
Directional Accuracy %      49.06
Accuracy on UP calls         0.47
Accuracy on DOWN calls       0.54
dtype: float64

,Date,Symbol,Close,Actual,Predicted,Confidence
0,2025-01-09,RELIANCE.NS,"1,244.01",0,0,0.53
1,2025-01-09,TCS.NS,"3,761.49",1,0,0.50
2,2025-01-09,^NSEI,"23,526.50",0,0,0.54
3,2025-01-10,RELIANCE.NS,"1,231.27",0,0,0.51
4,2025-01-10,TCS.NS,"3,972.72",1,1,0.53
5,2025-01-10,^NSEI,"23,431.50",0,0,0.50
6,2025-01-13,RELIANCE.NS,"1,229.23",0,1,0.52
7,2025-01-13,TCS.NS,"3,996.42",0,1,0.52
8,2025-01-13,^NSEI,"23,085.95",1,1,0.51
9,2025-01-14,RELIANCE.NS,"1,228.14",1,0,0.51


In [8]:
# accuracy per instrument
per_symbol = (preds.groupby("Symbol")
              .apply(lambda p: pd.Series({
                  "days": len(p),
                  "accuracy %": 100 * (p["Actual"] == p["Predicted"]).mean(),
                  "up calls": int(p["Predicted"].sum())}),
                  include_groups=False))
display(per_symbol.round(2))

,days,accuracy %,up calls
Symbol,,,
RELIANCE.NS,429.00,51.05,360.00
TCS.NS,429.00,45.69,368.00
^NSEI,424.00,50.47,251.00


## Phase 12 — Risk / volatility indicator

**Rule:** 10-day rolling annualised volatility of returns, i.e.
`std(returns[t−9..t]) × √252`, then thresholds fixed on the **training window only**
(33rd and 67th percentile):

```
vol <= p33           -> LOW VOLATILITY
p33   <  vol <= p67  -> MEDIUM VOLATILITY
vol   >  p67         -> HIGH VOLATILITY
```

Using training percentiles keeps the definition of "high" free of look-ahead bias.

In [9]:
from src.volatility import fit_thresholds, classify, label_series, volatility_report

thresholds = fit_thresholds(train)
print({k: round(v, 4) for k, v in thresholds.items()})
display(volatility_report(train, test, thresholds))

latest = (full.sort_values("Date").groupby("Symbol").tail(1)
          .assign(Level=lambda d: d["Volatility_10"].map(lambda v: classify(v, thresholds))))
display(latest[["Symbol", "Date", "Volatility_10", "Level"]].round(4))

{'low_max': 0.1472, 'medium_max': 0.224, 'median': 0.1852}


,Split,Symbol,LOW %,MEDIUM %,HIGH %,Mean vol %
0,train,,33.00,34.00,33.00,20.73
1,test,,41.40,29.70,28.90,18.34


,Symbol,Date,Volatility_10,Level
4285,TCS.NS,2026-09-25,0.28,HIGH
2142,RELIANCE.NS,2026-09-25,0.20,MEDIUM
6419,^NSEI,2026-09-25,0.12,LOW
